# 04 — Predictions & Simulation

In [1]:
import os, sys
# Move to project root so all relative paths (data/, models/, charts/) resolve correctly
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
from src.model import load_model, predict_proba
from src.features import build_match_features, FEATURE_NAMES
from src.twickenham import (
    BATH_STARTERS, EXETER_STARTERS, LEICESTER_STARTERS, NORTHAMPTON_STARTERS,
    substitute_twickenham_into_vector,
)
from src.simulate import TournamentTeams, run_tournament, simulate_scoreline
from src.charts import (plot_championship_win_pct, plot_scoreline_distribution,
                         plot_shap_waterfall, plot_twickenham_heatmap)

model = load_model()
raw = pd.read_csv('data/raw/current_season_stats.csv').set_index('team').to_dict('index')

# Semi 1: Bath (home) vs Exeter
v_semi1 = build_match_features(raw['Bath Rugby'], raw['Exeter Chiefs'], is_neutral=False)
p_bath = float(predict_proba(model, v_semi1.reshape(1, -1))[0])
print(f'Semi 1 — Bath win prob:      {p_bath:.1%}')
print(f'Semi 1 — Exeter win prob:    {1-p_bath:.1%}')

# Semi 2: Leicester (home) vs Northampton
v_semi2 = build_match_features(raw['Leicester Tigers'], raw['Northampton Saints'], is_neutral=False)
p_leicester = float(predict_proba(model, v_semi2.reshape(1, -1))[0])
print(f'Semi 2 — Leicester win prob: {p_leicester:.1%}')
print(f'Semi 2 — Northampton prob:   {1-p_leicester:.1%}')

Semi 1 — Bath win prob:      90.6%
Semi 1 — Exeter win prob:    9.4%
Semi 2 — Leicester win prob: 35.0%
Semi 2 — Northampton prob:   65.0%


In [2]:
# Final matchup probabilities (Twickenham features)
finals = {
    ('Bath Rugby',    'Leicester Tigers'):     substitute_twickenham_into_vector(BATH_STARTERS,    LEICESTER_STARTERS,    v_semi1),
    ('Bath Rugby',    'Northampton Saints'):   substitute_twickenham_into_vector(BATH_STARTERS,    NORTHAMPTON_STARTERS,  v_semi1),
    ('Exeter Chiefs', 'Leicester Tigers'):     substitute_twickenham_into_vector(EXETER_STARTERS,  LEICESTER_STARTERS,    v_semi2),
    ('Exeter Chiefs', 'Northampton Saints'):   substitute_twickenham_into_vector(EXETER_STARTERS,  NORTHAMPTON_STARTERS,  v_semi2),
}

match_probs = {
    ('Bath Rugby',       'Exeter Chiefs'):       p_bath,
    ('Leicester Tigers', 'Northampton Saints'):  p_leicester,
}
for matchup, vec in finals.items():
    match_probs[matchup] = float(predict_proba(model, vec.reshape(1, -1))[0])
    print(f'{matchup[0]} vs {matchup[1]}: {match_probs[matchup]:.1%}')

Bath Rugby vs Leicester Tigers: 50.8%
Bath Rugby vs Northampton Saints: 50.8%
Exeter Chiefs vs Leicester Tigers: 17.0%
Exeter Chiefs vs Northampton Saints: 17.0%


In [3]:
# Monte Carlo tournament
teams = TournamentTeams('Bath Rugby', 'Exeter Chiefs', 'Leicester Tigers', 'Northampton Saints')
championship_pcts = run_tournament(teams, match_probs, n=10_000)
print('\nChampionship win %:')
for team, pct in sorted(championship_pcts.items(), key=lambda x: -x[1]):
    print(f'  {team}: {pct:.1%}')


Championship win %:
  Bath Rugby: 46.3%
  Northampton Saints: 34.3%
  Leicester Tigers: 17.9%
  Exeter Chiefs: 1.5%


In [4]:
import matplotlib
matplotlib.use('Agg')

# Championship win % chart
plot_championship_win_pct(championship_pcts)

# Scoreline distributions for semi 1
outcome_semi1 = 'home' if p_bath > 0.5 else 'away'
home_scores = [simulate_scoreline(raw['Bath Rugby']['pts_scored_pg'],
                                   raw['Exeter Chiefs']['pts_scored_pg'],
                                   outcome_semi1)[0] for _ in range(5_000)]
away_scores = [simulate_scoreline(raw['Bath Rugby']['pts_scored_pg'],
                                   raw['Exeter Chiefs']['pts_scored_pg'],
                                   outcome_semi1)[1] for _ in range(5_000)]
plot_scoreline_distribution(home_scores, away_scores, 'Bath Rugby', 'Exeter Chiefs')

# SHAP waterfall for Semi 1
plot_shap_waterfall(model, v_semi1, FEATURE_NAMES, 'Bath vs Exeter (Semi 1)')

# Twickenham heatmap — only if starters are filled in
team_players = {
    'Bath Rugby': BATH_STARTERS,
    'Exeter Chiefs': EXETER_STARTERS,
    'Leicester Tigers': LEICESTER_STARTERS,
    'Northampton Saints': NORTHAMPTON_STARTERS,
}
if any(team_players.values()):
    plot_twickenham_heatmap(team_players)

/Users/yashsewpaul/code/prem-rugby-prediction/src/charts.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/Users/yashsewpaul/code/prem-rugby-prediction/src/charts.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/Users/yashsewpaul/code/prem-rugby-prediction/src/charts.py:87: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


/Users/yashsewpaul/code/prem-rugby-prediction/src/charts.py:113: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
